# 🏥 AfyaStock AI — Complete Dual-Model System
## Medicine Demand Forecasting & 30-Day Stock-Out Prevention

## 📋 Models:
1. **🎯 Classification**: Random Forest (Stock-out Prediction)
   - Target: `stockout_within_30_days`
   - Alert when medicine will run out in 30 days

2. **📈 Regression**: XGBoost (Demand Forecasting)
   - Target: `forecast_next_7_days`
   - Predict how much medicine will be needed

---
**⚠️ Important:** Dataset is synthetic - results demonstrate technical feasibility only. Must not be presented as validated Kenyan facility performance.

## 🔧 1. Setup & Installation

In [ ]:
# Install packages
!pip -q install xgboost shap openpyxl imbalanced-learn

In [ ]:
# Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
import os
from datetime import datetime
from math import sqrt

from IPython.display import display, HTML
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    # Classification metrics
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    # Regression metrics
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge
from imblearn.over_sampling import SMOTE
import shap

# Configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Color palette
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'success': '#73AB84',
    'warning': '#F18F01',
    'danger': '#C73E1D',
    'light': '#F5F5F5',
    'dark': '#1A1A2E',
    'demand': '#2E86AB',
    'stockout': '#C73E1D'
}

print("✅ Setup complete!")

## 📊 2. Data Loading & Exploration

In [ ]:
# Upload dataset
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
print("✅ Uploaded:", file_name)

In [ ]:
# Load dataset
from pathlib import Path
path = Path(file_name)
if path.suffix.lower() == ".csv":
    df = pd.read_csv(path)
elif path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(path)
else:
    raise ValueError("Upload a CSV or Excel file.")

print(f"📊 Shape: {df.shape}")
display(df.head())

In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip().str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("-", "_", regex=False)
)

In [ ]:
# Basic profile
print(f"📊 Rows: {len(df):,}")
print(f"📊 Columns: {len(df.columns)}")
print(f"📊 Duplicate complete rows: {df.duplicated().sum():,}")

### 📈 2.1 Data Overview Dashboard

In [ ]:
# ============================================================================
# 📊 DATA OVERVIEW DASHBOARD
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Dataset Overview Dashboard', fontsize=16, fontweight='bold', y=0.98)

# 1.1 Column Type Distribution
ax1 = axes[0, 0]
num_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(exclude=['number']).columns
data_composition = pd.DataFrame({
    'Category': ['Numeric', 'Categorical'],
    'Count': [len(num_cols), len(cat_cols)]
})
colors = ['#2E86AB', '#A23B72']
ax1.pie(data_composition['Count'], labels=data_composition['Category'], 
        autopct='%1.0f%%', colors=colors, startangle=90, explode=(0.05, 0))
ax1.set_title('Column Type Distribution', fontsize=12, fontweight='bold')

# 1.2 Data Completeness
ax2 = axes[0, 1]
completeness = (1 - df.isna().mean()) * 100
completeness_sorted = completeness.sort_values()
top_features = completeness_sorted.tail(15)
ax2.barh(top_features.index, top_features.values, color='#2E86AB', alpha=0.8)
ax2.set_xlabel('Completeness (%)')
ax2.set_title('Top 15 Features by Completeness', fontsize=12, fontweight='bold')
ax2.set_xlim([0, 105])
ax2.axvline(x=95, color='red', linestyle='--', alpha=0.5, label='95% Threshold')
ax2.legend()

# 1.3 Records by Year
ax3 = axes[1, 0]
year_counts = df['year'].value_counts().sort_index()
bars = ax3.bar(year_counts.index, year_counts.values, color='#73AB84')
ax3.set_xlabel('Year')
ax3.set_ylabel('Number of Records')
ax3.set_title('Records by Year', fontsize=12, fontweight='bold')
for bar, count in zip(bars, year_counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, 
             f'{count:,}', ha='center', va='bottom', fontsize=9)

# 1.4 Records by Month
ax4 = axes[1, 1]
month_counts = df['month'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax4.plot(month_names, month_counts.values, marker='o', color='#F18F01', 
         linewidth=2, markersize=8)
ax4.fill_between(month_names, month_counts.values, alpha=0.2, color='#F18F01')
ax4.set_xlabel('Month')
ax4.set_ylabel('Number of Records')
ax4.set_title('Seasonal Distribution of Records', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🧹 3. Data Cleaning

In [ ]:
# ============================================================================
# 🧹 DATA CLEANING
# ============================================================================

# Clean categorical columns
categorical_candidates = [
    "facility_type","facility_size","facility_location_type",
    "medicine","medicine_category","dosage_form","stock_out_flag",
    "supplier_type","supplier_reliability","purchase_order_frequency",
    "orders_cancelled","reorder_flag","expiry_risk","demand_shock"
]
categorical_columns = [c for c in categorical_candidates if c in df.columns]

for col in categorical_columns:
    df[col] = (
        df[col].astype("string").str.strip().str.lower()
        .replace({"": pd.NA, "nan": pd.NA, "none": pd.NA})
    )

# Convert numeric columns
numeric_candidates = [
    "registered_patients","outpatient_visits","prescriptions_issued",
    "previous_day_demand","previous_week_demand","demand_7_days_avg",
    "demand_14_days_avg","demand_30_days_avg","demand_90_days_avg",
    "expected_demand","opening_stock","stock_received","stock_adjustments",
    "dispensed_quantity","closing_stock","stock_out_days","lost_demand_units",
    "lead_time_days","purchase_order_quantity","order_delay_days",
    "safety_stock","reorder_point","inventory_turnover","expired_quantity",
    "damaged_quantity","inventory_value_kes","unit_cost_kes",
    "forecast_next_7_days","forecast_next_14_days","forecast_next_30_days",
    "forecast_error","forecast_error_percentage","stockout_risk_score"
]
numeric_columns = [c for c in numeric_candidates if c in df.columns]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("✅ Data cleaning complete!")

## 📈 4. Exploratory Data Analysis

In [ ]:
# ============================================================================
# 🎯 TARGET DISTRIBUTION ANALYSIS
# ============================================================================

target_class = "stockout_within_30_days"
target_reg = "forecast_next_7_days"

df[target_class] = pd.to_numeric(df[target_class], errors="coerce")

# Classification Target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎯 Target Distribution Analysis', fontsize=14, fontweight='bold')

target_counts = df[target_class].value_counts()
labels = ['No Stock-out', 'Stock-out']
colors_bar = ['#73AB84', '#C73E1D']
bars = axes[0].bar(labels, target_counts.values, color=colors_bar, edgecolor='black', linewidth=1)
axes[0].set_ylabel('Count')
axes[0].set_title('Stock-out Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, target_counts.max() * 1.1)

for bar, count, label in zip(bars, target_counts.values, labels):
    percentage = count / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{count:,}
({percentage:.1f}%)', ha='center', va='bottom', fontsize=10)

# Regression Target
ax2 = axes[1]
forecast_data = df[target_reg].dropna()
ax2.hist(forecast_data, bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Forecasted Demand (Next 7 Days)')
ax2.set_ylabel('Frequency')
ax2.set_title('Demand Forecast Distribution', fontsize=12, fontweight='bold')
ax2.axvline(forecast_data.mean(), color='red', linestyle='--', 
            label=f'Mean: {forecast_data.mean():.1f}')
ax2.axvline(forecast_data.median(), color='orange', linestyle='--', 
            label=f'Median: {forecast_data.median():.1f}')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📊 Classification Target:")
print(f"   No Stock-out: {target_counts[0]:,} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"   Stock-out:    {target_counts[1]:,} ({target_counts[1]/len(df)*100:.1f}%)")

print(f"\n📊 Regression Target (Next 7-Day Forecast):")
print(f"   Mean: {forecast_data.mean():.2f}")
print(f"   Median: {forecast_data.median():.2f}")
print(f"   Std: {forecast_data.std():.2f}")
print(f"   Min: {forecast_data.min():.2f}")
print(f"   Max: {forecast_data.max():.2f}")

In [ ]:
# ============================================================================
# 💊 MEDICINE & FACILITY ANALYSIS
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('💊 Medicine & Facility Analysis', fontsize=16, fontweight='bold')

# Top Medicines by Stock-out Rate
ax1 = axes[0, 0]
medicine_data = df.groupby('medicine')[target_class].mean().sort_values(ascending=False).head(15) * 100
bars = ax1.barh(medicine_data.index, medicine_data.values, color='#C73E1D', alpha=0.8)
ax1.set_xlabel('Stock-out Rate (%)')
ax1.set_title('Top 15 Medicines by Stock-out Rate', fontsize=12, fontweight='bold')
ax1.axvline(x=df[target_class].mean()*100, color='blue', linestyle='--', 
            alpha=0.7, label=f'Overall: {df[target_class].mean()*100:.1f}%')
ax1.legend()

for i, (idx, val) in enumerate(medicine_data.items()):
    ax1.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=8)

# Medicine Demand
ax2 = axes[0, 1]
medicine_demand = df.groupby('medicine')[target_reg].mean().sort_values(ascending=False).head(15)
bars = ax2.barh(medicine_demand.index, medicine_demand.values, color='#2E86AB', alpha=0.8)
ax2.set_xlabel('Average Demand (Next 7 Days)')
ax2.set_title('Top 15 Medicines by Demand', fontsize=12, fontweight='bold')
for i, (idx, val) in enumerate(medicine_demand.items()):
    ax2.text(val + 1, i, f'{val:.1f}', va='center', fontsize=8)

# Facility Type Stock-out Rates
ax3 = axes[1, 0]
facility_data = df.groupby('facility_type')[target_class].mean().sort_values(ascending=False) * 100
colors_fac = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(facility_data)))
bars = ax3.bar(facility_data.index, facility_data.values, color=colors_fac)
ax3.set_ylabel('Stock-out Rate (%)')
ax3.set_title('Stock-out Rate by Facility Type', fontsize=12, fontweight='bold')
ax3.axhline(y=df[target_class].mean()*100, color='red', linestyle='--', 
            alpha=0.7, label='Overall Average')
ax3.legend()
ax3.tick_params(axis='x', rotation=45, ha='right')

for bar, val in zip(bars, facility_data.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# Medicine Category Distribution
ax4 = axes[1, 1]
category_counts = df['medicine_category'].value_counts().head(10)
bars = ax4.barh(category_counts.index, category_counts.values, color='#A23B72', alpha=0.8)
ax4.set_xlabel('Count')
ax4.set_title('Top 10 Medicine Categories', fontsize=12, fontweight='bold')
for i, (idx, val) in enumerate(category_counts.items()):
    ax4.text(val + 100, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 🎯 5. Feature Engineering

In [ ]:
# ============================================================================
# 🎯 FEATURE SELECTION & LEAKAGE PREVENTION
# ============================================================================

leakage_cols = ['record_id', 'date', 'stock_out_flag', 'stock_out_days',
                'stockout_risk_score', 'stockout_risk_category']

# Classification features
X_clf = df.drop(columns=[target_class] + leakage_cols, errors='ignore')
y_clf = df[target_class]

# Regression features (excluding target-like columns)
reg_exclude = [target_reg, 'forecast_next_14_days', 'forecast_next_30_days', 
               'forecast_error', 'forecast_error_percentage']
X_reg = df.drop(columns=[target_class] + leakage_cols + reg_exclude, errors='ignore')
y_reg = df[target_reg]

print(f"📊 Classification Features: {X_clf.shape}")
print(f"📊 Regression Features: {X_reg.shape}")

In [ ]:
# ============================================================================
# 🔀 TRAIN/TEST SPLIT
# ============================================================================

# Classification split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=RANDOM_STATE
)

# Regression split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

print(f"📊 Classification - Train: {len(X_train_clf):,}, Test: {len(X_test_clf):,}")
print(f"📊 Regression - Train: {len(X_train_reg):,}, Test: {len(X_test_reg):,}")

In [ ]:
# ============================================================================
# 🔧 PREPROCESSING PIPELINE
# ============================================================================

def build_preprocessor(X_train):
    """Build and fit a preprocessor for the given data"""
    num_cols = X_train.select_dtypes(include=['number']).columns
    cat_cols = X_train.select_dtypes(exclude=['number']).columns
    
    # Convert pandas StringDtype to object
    X_train[cat_cols] = X_train[cat_cols].astype(object).replace({pd.NA: np.nan})
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])
    
    return preprocessor, num_cols, cat_cols

# Classification preprocessor
preprocessor_clf, num_cols_clf, cat_cols_clf = build_preprocessor(X_train_clf.copy())
X_train_clf_processed = preprocessor_clf.fit_transform(X_train_clf)
X_test_clf_processed = preprocessor_clf.transform(X_test_clf)

# Regression preprocessor
preprocessor_reg, num_cols_reg, cat_cols_reg = build_preprocessor(X_train_reg.copy())
X_train_reg_processed = preprocessor_reg.fit_transform(X_train_reg)
X_test_reg_processed = preprocessor_reg.transform(X_test_reg)

print("✅ Preprocessors built and fitted!")

In [ ]:
# ============================================================================
# 🔄 SMOTE OVERSAMPLING (Classification Only)
# ============================================================================

smote = SMOTE(random_state=RANDOM_STATE)
X_train_clf_balanced, y_train_clf_balanced = smote.fit_resample(
    X_train_clf_processed, y_train_clf
)

print(f"✅ Classification balanced: {len(X_train_clf_balanced):,} samples")
print(f"   Class distribution: {y_train_clf_balanced.value_counts().to_dict()}")

## 🤖 6. Model Training

In [ ]:
# ============================================================================
# 🎯 MODEL 1: STOCK-OUT CLASSIFICATION (RANDOM FOREST)
# ============================================================================

print("\n" + "="*60)
print("🎯 TRAINING MODEL 1: STOCK-OUT CLASSIFICATION")
print("="*60)

clf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)
clf_model.fit(X_train_clf_balanced, y_train_clf_balanced)

# Predictions
y_pred_clf = clf_model.predict(X_test_clf_processed)
y_proba_clf = clf_model.predict_proba(X_test_clf_processed)[:, 1]

# Evaluate
print("\n📊 Classification Performance:")
print("="*60)
print(classification_report(y_test_clf, y_pred_clf))
print(f"ROC-AUC: {roc_auc_score(y_test_clf, y_proba_clf):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test_clf, y_proba_clf):.4f}")

In [ ]:
# ============================================================================
# 🎯 OPTIMAL THRESHOLD (Classification)
# ============================================================================

precision, recall, thresholds = precision_recall_curve(y_test_clf, y_proba_clf)

f1_scores = []
for threshold in thresholds:
    y_pred = (y_proba_clf >= threshold).astype(int)
    f1_scores.append(f1_score(y_test_clf, y_pred, zero_division=0))

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"\n🎯 Optimal Threshold: {optimal_threshold:.4f}")
print(f"📊 Max F1-Score: {optimal_f1:.4f}")

In [ ]:
# ============================================================================
# 📈 MODEL 2: DEMAND FORECASTING (XGBOOST REGRESSION)
# ============================================================================

print("\n" + "="*60)
print("📈 TRAINING MODEL 2: DEMAND FORECASTING")
print("="*60)

reg_model = XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
reg_model.fit(X_train_reg_processed, y_train_reg)

# Predictions
y_pred_reg = reg_model.predict(X_test_reg_processed)

# Evaluate
print("\n📊 Regression Performance:")
print("="*60)
print(f"R² Score:        {r2_score(y_test_reg, y_pred_reg):.4f}")
print(f"MAE:             {mean_absolute_error(y_test_reg, y_pred_reg):.4f}")
print(f"RMSE:            {sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}")
print(f"MAPE:            {(np.abs((y_test_reg - y_pred_reg) / y_test_reg).mean() * 100):.2f}%")

## 📊 7. Model Evaluation Dashboard

In [ ]:
# ============================================================================
# 📊 CLASSIFICATION PERFORMANCE DASHBOARD
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('🎯 Stock-Out Classification - Performance Dashboard', fontsize=16, fontweight='bold')

# ROC Curve
ax1 = axes[0, 0]
fpr, tpr, _ = roc_curve(y_test_clf, y_proba_clf)
auc = roc_auc_score(y_test_clf, y_proba_clf)
ax1.plot(fpr, tpr, color='#2E86AB', linewidth=3, label=f'ROC (AUC={auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR Curve
ax2 = axes[0, 1]
ap = average_precision_score(y_test_clf, y_proba_clf)
ax2.plot(recall, precision, color='#A23B72', linewidth=3, label=f'PR (AP={ap:.4f})')
ax2.scatter(recall[optimal_idx], precision[optimal_idx], 
            color='red', s=150, zorder=5, 
            label=f'Optimal\nThreshold={optimal_threshold:.3f}\nF1={optimal_f1:.3f}')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

# Confusion Matrix
ax3 = axes[1, 0]
cm = confusion_matrix(y_test_clf, y_pred_clf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Stock-out', 'Stock-out'],
            yticklabels=['No Stock-out', 'Stock-out'],
            ax=ax3, cbar=False, annot_kws={'size': 14})
ax3.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
ax3.set_ylabel('Actual')
ax3.set_xlabel('Predicted')

# Metrics Summary
ax4 = axes[1, 1]
ax4.axis('off')
metrics_summary = [
    ('Accuracy', accuracy_score(y_test_clf, y_pred_clf)),
    ('Precision', precision_score(y_test_clf, y_pred_clf, zero_division=0)),
    ('Recall', recall_score(y_test_clf, y_pred_clf, zero_division=0)),
    ('F1-Score', f1_score(y_test_clf, y_pred_clf, zero_division=0)),
    ('ROC-AUC', roc_auc_score(y_test_clf, y_proba_clf)),
    ('PR-AUC', average_precision_score(y_test_clf, y_proba_clf))
]

table_data = [[m, f'{v:.3f}'] for m, v in metrics_summary]
table = ax4.table(cellText=table_data, 
                  colLabels=['Metric', 'Value'],
                  loc='center',
                  cellLoc='center',
                  colWidths=[0.4, 0.3],
                  bbox=[0.1, 0.2, 0.8, 0.6])

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 1.5)

# Color code cells
for i, (_, val) in enumerate(metrics_summary):
    color = '#D4E6F1' if val >= 0.7 else '#FADBD8'
    table[(i+1, 1)].set_facecolor(color)

ax4.set_title('Performance Summary', fontsize=12, fontweight='bold', y=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 📈 REGRESSION PERFORMANCE DASHBOARD
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('📈 Demand Forecasting - Performance Dashboard', fontsize=16, fontweight='bold')

# Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(y_test_reg, y_pred_reg, alpha=0.3, color='#2E86AB')
ax1.plot([y_test_reg.min(), y_test_reg.max()], 
         [y_test_reg.min(), y_test_reg.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Demand')
ax1.set_ylabel('Predicted Demand')
ax1.set_title('Actual vs Predicted Demand', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals Distribution
ax2 = axes[0, 1]
residuals = y_test_reg - y_pred_reg
ax2.hist(residuals, bins=50, color='#A23B72', alpha=0.7, edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residual')
ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold')

# Error Metrics
ax3 = axes[1, 0]
ax3.axis('off')
error_metrics = [
    ('R² Score', r2_score(y_test_reg, y_pred_reg)),
    ('MAE', mean_absolute_error(y_test_reg, y_pred_reg)),
    ('RMSE', sqrt(mean_squared_error(y_test_reg, y_pred_reg))),
    ('MAPE', (np.abs((y_test_reg - y_pred_reg) / y_test_reg).mean() * 100))
]

table_data = [[m, f'{v:.4f}'] for m, v in error_metrics]
table = ax3.table(cellText=table_data, 
                  colLabels=['Metric', 'Value'],
                  loc='center',
                  cellLoc='center',
                  colWidths=[0.4, 0.3],
                  bbox=[0.1, 0.1, 0.8, 0.8])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 1.5)
ax3.set_title('Error Metrics', fontsize=12, fontweight='bold', y=0.9)

# Residuals vs Predicted
ax4 = axes[1, 1]
ax4.scatter(y_pred_reg, residuals, alpha=0.3, color='#F18F01')
ax4.axhline(0, color='red', linestyle='--', linewidth=2)
ax4.set_xlabel('Predicted Demand')
ax4.set_ylabel('Residual')
ax4.set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### ⭐ 7.1 Feature Importance Analysis

In [ ]:
# ============================================================================
# ⭐ FEATURE IMPORTANCE (Both Models)
# ============================================================================

# Get feature names
cat_feature_names_clf = preprocessor_clf.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols_clf)
all_feature_names_clf = list(num_cols_clf) + list(cat_feature_names_clf)

cat_feature_names_reg = preprocessor_reg.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols_reg)
all_feature_names_reg = list(num_cols_reg) + list(cat_feature_names_reg)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('⭐ Feature Importance Comparison', fontsize=16, fontweight='bold')

# Classification Feature Importance
ax1 = axes[0]
importances_clf = clf_model.feature_importances_
indices_clf = np.argsort(importances_clf)[-15:]
top_features_clf = [all_feature_names_clf[i] for i in indices_clf]
top_importances_clf = importances_clf[indices_clf]

colors_imp = plt.cm.Blues(np.linspace(0.3, 0.9, len(top_features_clf)))
bars = ax1.barh(range(len(top_features_clf)), top_importances_clf, color=colors_imp[::-1])
ax1.set_yticks(range(len(top_features_clf)))
ax1.set_yticklabels(top_features_clf)
ax1.set_xlabel('Relative Importance')
ax1.set_title('Classification (Stock-Out)', fontsize=12, fontweight='bold')
for i, (bar, val) in enumerate(zip(bars, top_importances_clf)):
    ax1.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=8)

# Regression Feature Importance
ax2 = axes[1]
importances_reg = reg_model.feature_importances_
indices_reg = np.argsort(importances_reg)[-15:]
top_features_reg = [all_feature_names_reg[i] for i in indices_reg]
top_importances_reg = importances_reg[indices_reg]

colors_imp = plt.cm.Reds(np.linspace(0.3, 0.9, len(top_features_reg)))
bars = ax2.barh(range(len(top_features_reg)), top_importances_reg, color=colors_imp[::-1])
ax2.set_yticks(range(len(top_features_reg)))
ax2.set_yticklabels(top_features_reg)
ax2.set_xlabel('Relative Importance')
ax2.set_title('Regression (Demand Forecast)', fontsize=12, fontweight='bold')
for i, (bar, val) in enumerate(zip(bars, top_importances_reg)):
    ax2.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 💾 8. Model Persistence

In [ ]:
# ============================================================================
# 💾 SAVE MODELS
# ============================================================================

# Create models directory
os.makedirs('afyastock_models', exist_ok=True)

# Save classification model
joblib.dump(clf_model, 'afyastock_models/classifier_stockout.joblib')
joblib.dump(preprocessor_clf, 'afyastock_models/preprocessor_clf.joblib')

# Save regression model
joblib.dump(reg_model, 'afyastock_models/regressor_demand.joblib')
joblib.dump(preprocessor_reg, 'afyastock_models/preprocessor_reg.joblib')

# Save metadata
metadata = {
    'created': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'classification': {
        'model': 'Random Forest',
        'target': target_class,
        'optimal_threshold': optimal_threshold,
        'n_estimators': 200,
        'max_depth': 15,
        'performance': {
            'accuracy': accuracy_score(y_test_clf, y_pred_clf),
            'precision': precision_score(y_test_clf, y_pred_clf, zero_division=0),
            'recall': recall_score(y_test_clf, y_pred_clf, zero_division=0),
            'f1_score': f1_score(y_test_clf, y_pred_clf, zero_division=0),
            'roc_auc': roc_auc_score(y_test_clf, y_proba_clf),
            'pr_auc': average_precision_score(y_test_clf, y_proba_clf)
        }
    },
    'regression': {
        'model': 'XGBoost',
        'target': target_reg,
        'performance': {
            'r2': r2_score(y_test_reg, y_pred_reg),
            'mae': mean_absolute_error(y_test_reg, y_pred_reg),
            'rmse': sqrt(mean_squared_error(y_test_reg, y_pred_reg)),
            'mape': (np.abs((y_test_reg - y_pred_reg) / y_test_reg).mean() * 100)
        }
    }
}

with open('afyastock_models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✅ Models saved to 'afyastock_models/'")
print("\n📁 Contents:")
for f in os.listdir('afyastock_models'):
    print(f"   - {f}")

## 📋 9. Summary Report

In [ ]:
# ============================================================================
# 📋 FINAL SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("📋 AFYASTOCK AI - COMPLETE SYSTEM REPORT")
print("="*80)

print("\n" + "="*80)
print("🎯 CLASSIFICATION MODEL: STOCK-OUT PREDICTION")
print("="*80)
print(f"""
Model:              Random Forest
Target:             {target_class}
Performance:
  Accuracy:         {accuracy_score(y_test_clf, y_pred_clf):.4f}
  Precision:        {precision_score(y_test_clf, y_pred_clf, zero_division=0):.4f}
  Recall:           {recall_score(y_test_clf, y_pred_clf, zero_division=0):.4f}
  F1-Score:         {f1_score(y_test_clf, y_pred_clf, zero_division=0):.4f}
  ROC-AUC:          {roc_auc_score(y_test_clf, y_proba_clf):.4f}
  PR-AUC:           {average_precision_score(y_test_clf, y_proba_clf):.4f}
Optimal Threshold:  {optimal_threshold:.4f}
""")

print("="*80)
print("📈 REGRESSION MODEL: DEMAND FORECASTING")
print("="*80)
print(f"""
Model:              XGBoost
Target:             {target_reg}
Performance:
  R² Score:         {r2_score(y_test_reg, y_pred_reg):.4f}
  MAE:              {mean_absolute_error(y_test_reg, y_pred_reg):.4f}
  RMSE:             {sqrt(mean_squared_error(y_test_reg, y_pred_reg)):.4f}
  MAPE:             {(np.abs((y_test_reg - y_pred_reg) / y_test_reg).mean() * 100):.2f}%
""")

print("="*80)
print("📁 MODEL FILES")
print("="*80)
print("""
afyastock_models/
├── classifier_stockout.joblib      # Random Forest (Stock-out)
├── preprocessor_clf.joblib          # Classification preprocessor
├── regressor_demand.joblib          # XGBoost (Demand Forecast)
├── preprocessor_reg.joblib          # Regression preprocessor
└── model_metadata.pkl               # System metadata
""")

print("="*80)
print("✅ System Complete!")
print("="*80)"

## 🔍 Quick Inference Example

In [ ]:
# ============================================================================
# 🔍 QUICK INFERENCE EXAMPLE
# ============================================================================

print("\n🔍 Quick Inference Example:")

# Load models
loaded_clf = joblib.load('afyastock_models/classifier_stockout.joblib')
loaded_reg = joblib.load('afyastock_models/regressor_demand.joblib')
loaded_preprocessor_clf = joblib.load('afyastock_models/preprocessor_clf.joblib')
loaded_preprocessor_reg = joblib.load('afyastock_models/preprocessor_reg.joblib')

# Sample prediction (Classification)
sample_idx = X_test_clf.index[0]
sample_clf = X_test_clf.iloc[0:1]
sample_clf_processed = loaded_preprocessor_clf.transform(sample_clf)
pred_clf = loaded_clf.predict(sample_clf_processed)[0]
proba_clf = loaded_clf.predict_proba(sample_clf_processed)[0, 1]

# Sample prediction (Regression)
sample_reg = X_test_reg.iloc[0:1]
sample_reg_processed = loaded_preprocessor_reg.transform(sample_reg)
pred_reg = loaded_reg.predict(sample_reg_processed)[0]

print(f"\n📊 Sample Record ID: {df.iloc[sample_idx]['record_id']}")
print(f"   Medicine: {df.iloc[sample_idx]['medicine']}")
print(f"\n🎯 Stock-Out Prediction:")
print(f"   Prediction: {'⚠️ Stock-out' if pred_clf == 1 else '✅ No Stock-out'}")
print(f"   Probability: {proba_clf:.4f}")
print(f"   Decision: {'🔴 ALERT' if proba_clf >= optimal_threshold else '✅ OK'}")
print(f"\n📈 Demand Forecast:")
print(f"   Predicted Demand (Next 7 Days): {pred_reg:.2f} units")

print("\n" + "="*80)